# Vision Module sync calibration
Upload a CSV exported from consented labeled clips with `clip_id,ts_ms,audio_rms,mouth_aperture,face_found,landmark_quality,label`. This notebook estimates best lag correlation over ±400 ms and compares real and fake distributions. Keep a holdout set out of threshold selection.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
csv_path = Path('/content/sync_measurements.csv')
df = pd.read_csv(csv_path)
df.head()

In [ ]:
def best_lag_corr(a, b, max_lag=4):
    best = -1.0
    for lag in range(-max_lag, max_lag + 1):
        x = a[max(0, lag): len(a) + min(0, lag)]
        y = b[max(0, -lag): len(b) + min(0, -lag)]
        if len(x) > 5 and x.std() > 1e-6 and y.std() > 1e-6:
            best = max(best, float(np.corrcoef(x, y)[0, 1]))
    return best
def clip_scores(g):
    g = g.sort_values('ts_ms').copy()
    # The production detector scores 8-second windows; this summarizes each clip for exploration.
    return best_lag_corr(g.audio_rms.to_numpy(), g.mouth_aperture.to_numpy())
scores = df.groupby(['clip_id','label']).apply(clip_scores, include_groups=False).rename('correlation').reset_index()
scores.groupby('label').correlation.describe()

In [ ]:
# Revisit these values against separate holdout clips before recording them in calibration.json.
scores.boxplot(column='correlation', by='label');